# Qwen3.8-27B on a free Kaggle TPU v5e-8 🚀

This notebook serves **Qwen3.8-27B (bf16)** with vLLM on Kaggle's free TPU and gives you a
public OpenAI-compatible endpoint you can use from anywhere — your laptop, Claude Code,
Codex CLI, opencode, or plain `curl`.

Measured on this exact setup: **~104 tok/s** single-stream decode (verified-lossless MTP speculative decoding), **~900 tok/s** aggregate across 16 streams, **10,000+ tok/s prefill**,
context up to the model's native **262,144 tokens**.

## Before you run — 3 clicks in the right sidebar
1. **Accelerator → TPU VM v5e-8** (Session options)
2. **Internet → ON** (Session options — needed for pip and the tunnel)
3. **Add Input** → search and attach these two datasets:
   - `rahim3/qwen3-8-27b-bf16` — the model weights (skips a big download)
   - `rahim3/qwen38-xla-cache-v5e8` — pre-built XLA compile cache (halves startup time)

Then run the cells top to bottom. The last cell keeps running **on purpose** — that's your
server. The endpoint URL + API key appear in its output after ~15–35 minutes.


In [ ]:
%%writefile serve_config.json
{
  "max_model_len": 262144,
  "max_num_seqs": 4,
  "mtp_tokens": 3,
  "reasoning_effort_default": "xhigh",
  "keepalive_min": 90
}


### Configuration
The defaults above are the sweet spot from our benchmarks. Things you might change:
- `max_model_len`: defaults to the native `262144`; for many parallel streams use
  `131072` with `"max_num_seqs": 16` (~900 tok/s aggregate)
- `mtp_tokens`: `3` = MTP speculative decoding (+34% decode). The script auto-applies a
  bundled fix (port of tpu-inference PR #3178) that makes it lossless on TPU —
  verified 12/12 greedy exact-match. `0` disables
- `reasoning_effort_default`: `"xhigh"` (default), `"medium"`, or `"low"` — clients can
  still override per request with `chat_template_kwargs`
- `keepalive_min`: the server shuts itself down after this long — raise it (up to ~480)
  when you want a long-lived endpoint; the default keeps a test run cheap on your TPU quota


### The serving script
The next cell writes the full serving script (same one the CLI launcher pushes —
see the [kaggle-tpu-lab repo](https://github.com/ARahim3/kaggle-tpu-lab) for the source and docs).


In [ ]:
%%writefile serve_qwen38.py
"""
Serve Qwen3.8-27B (bf16) on a Kaggle TPU v5e-8 with vLLM.

This script is pushed to Kaggle as a script kernel by ../launch.py, which fills
in the CFG line below. It also runs standalone with defaults (e.g. pasted into
a Kaggle notebook/script in the UI) — then it just prints instead of using ntfy.

What it does:
  1. installs vllm-tpu (pinned; includes the gated-DeltaNet TPU kernels)
  2. restores the XLA compile cache from a mounted dataset (halves cold start)
  3. finds the weights (mounted Kaggle dataset, or downloads from HF to /tmp)
  4. starts the vLLM OpenAI-compatible server (TP=8, optional MTP spec-decode)
  5. opens a public cloudflared tunnel and prints/publishes the endpoint
  6. keeps serving until KEEPALIVE_MIN elapses, then exits cleanly
"""
import collections
import glob
import json
import os
import re
import secrets
import subprocess
import sys
import threading
import time
import urllib.request
from pathlib import Path

CFG = None  # __LAUNCHER_CONFIG__  (launch.py replaces this line)

DEFAULTS = {
    "vllm_tpu_version": "0.28.0",
    "weights_dataset": "rahim3/qwen3-8-27b-bf16",     # HF mirror of Qwen/Qwen3.8-27B
    "hf_model_id": "Qwen/Qwen3.8-27B",                # fallback download source
    "max_model_len": 262144,       # native context; drop to 131072 + max_num_seqs 16 for throughput
    "max_num_seqs": 4,
    "mtp_tokens": 3,               # MTP spec decoding (+34% decode). Stock vllm-tpu
                                   # 0.28.0 corrupts outputs with it (missing GDN state
                                   # rollback); we apply patches/mtp-rollback-v0280.diff
                                   # (a port of upstream PR #3178) before serving —
                                   # verified lossless, 12/12 greedy exact-match.
    "reasoning_effort_default": "xhigh",   # server-side default: xhigh | medium | low
    "tool_call_parser": "qwen3_coder",  # matches Qwen3.8's XML tool format; "" disables
    "keepalive_min": 480,          # auto-shutdown guard (Kaggle TPU caps at 9h anyway)
    "api_key": "",                 # generated if empty
    "ntfy_topic": "",              # optional: publish progress to ntfy.sh/<topic>
    "served_model_name": "qwen3.8-27b",
}
CFG = {**DEFAULTS, **(CFG or {})}
# Notebook flow: drop overrides in a serve_config.json next to this script.
_cfg_file = Path("serve_config.json")
if _cfg_file.exists():
    CFG.update(json.loads(_cfg_file.read_text()))
if not CFG["api_key"]:
    CFG["api_key"] = "sk-" + secrets.token_hex(16)

PORT = 8000
XLA_CACHE = "/tmp/xla_cache"
os.environ["HF_HOME"] = "/tmp/hf"                 # /kaggle/working is only ~21 GB
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
os.environ["VLLM_XLA_CACHE_PATH"] = XLA_CACHE


def log(*parts):
    print(time.strftime("[%H:%M:%S]"), " ".join(str(p) for p in parts), flush=True)


def publish(phase, **extra):
    """Progress event: always logged; also pushed to ntfy if a topic is set."""
    log(f"PHASE {phase}", json.dumps(extra) if extra else "")
    if not CFG["ntfy_topic"]:
        return
    try:
        body = {"topic": CFG["ntfy_topic"], "title": f"kaggle-tpu-lab {phase}",
                "message": json.dumps({"phase": phase, **extra})}
        req = urllib.request.Request("https://ntfy.sh", data=json.dumps(body).encode(),
                                     headers={"Content-Type": "application/json"})
        urllib.request.urlopen(req, timeout=10)
    except Exception as e:
        log(f"(ntfy publish failed: {e})")


def run_stream(cmd, tag):
    p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    def pump():
        for line in p.stdout:
            line = line.rstrip()
            if line:
                log(f"[{tag}] {line[:400]}")
    threading.Thread(target=pump, daemon=True).start()
    return p


def find_input(*patterns):
    """Datasets mount at /kaggle/input/<slug> (UI) or /kaggle/input/datasets/<owner>/<slug> (API push)."""
    for pat in patterns:
        hits = glob.glob(f"/kaggle/input/{pat}") + glob.glob(f"/kaggle/input/datasets/*/{pat}")
        if hits:
            return hits[0]
    return None


# ---------------- 1. install ----------------
publish("install", vllm_tpu=CFG["vllm_tpu_version"])
t = time.time()
rc = run_stream([sys.executable, "-m", "pip", "install", "-q",
                 f"vllm-tpu=={CFG['vllm_tpu_version']}"], "pip").wait()
if rc != 0:
    publish("failed", step="pip-install")
    sys.exit(1)
# Kaggle's preinstalled torchaudio is ABI-incompatible with the torch that
# vllm-tpu pins (crashes vLLM at import). Text serving doesn't need it.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchaudio"],
               check=False)
publish("installed", secs=int(time.time() - t))

# gzip+base64 of patches/mtp-rollback-v0280.diff; regenerated by tools/embed_patch.py
MTP_PATCH_B64 = "H4sIAPLZlWoC/+09a3PjOHLf/StQnkodFVEaPfzUnbbWNzN3t7mZWdeM95KK46IpEZJoU6SWpOTxXa4qPyK/ML8k3XiQAAhSlHcmuUtlateWSaABNBr9QncrCBcL0ustw5z4r/PN1gvjBU1pPKevI/+ZptnrebJeJ/FrP89pnIdJ7K1p7gd+7vc3z2R2eJ+jMA7oF3I+G/rB5Wm/HywGJ/OzORkOBmcnJ0e9Xu8lMznqdrsvms3335Pe6al7Rrrw84LAn/PIzzJy/eb6g2g2OSLy3/HPW5o+e1nup7kXJfNjV3mXUnib5V4QZnkazrY4ltZg7a9nPnbOKcwyCOc0g/dd832U5F5K/cBLFouM5rY2e4fazDfy7zvxG1ftLUIaBdn09njjBwENvHi7RmAwBuvjzf35inobfwlTg46InYsBYudi5A7PS/RcSWRKJDnJ7IHO845A1ivyMYkpWSQpWScBjTLyFOarZJsTtgDCt+bXZLNNaY8/ku2SbRQQP8oSCWmbUZKvwoywybskiaNnsnqepWEgO9EvNJ2H0A7oOE8C/7kv1lzF+IQ8+F/6V2nqP5N/57Ocsl8cya+IwzshZmawxY+Z2yFhnI9H5L/+4z/F9Lsk29A5CegcxmcTcgnMgzJAfXJN0x5uowTJJkBwTwnfU+IgeH8+p5ucBqRHhmSRJmtYJgXUZDnZ0TRcPENHuulM2OPfv/0owT3SNKYRwMu2a5rxnrBt88dNAhMl9zM/o4yMYJ58vHvixwF5SsMc2sf0SUIqewkwZd/7PvkBTypMb/ZMNqvnLJz7EWFgswQRnW3TXbijmQQG8wF4wTYK42WfP6yh6P17MCb/SIKNl4V/poej/334WEC6tx2Ve5fAB1gD0lQK6M7ocg30TObJFlHhRxHQaxwkTzQo9hDhAFfJiMPHzWAaYpt426zjImbEZhW7tI0zNpIJcM3mDhBYj1sO9O6Wg7wjm5Quwi+wf+z1zM/nKw2ptnXV45XIwwR8rYfICxfhnBSMkLd9WtGYxLC/MOcYNpHPAXAxT+KcfsnJxk8BNTQKs7U4YMAzJiqvNIYN2oqXZRB7BYPeL1nM5kKo+BeXl7PA7/fn/mg2vxy3FyoVgPvlSaULMsvhqXtOukMUKN9/f0R+CzxqSdkO5kk6XxGtkzdPUiCBDWOTSx85AXCz3I+BQRSNyM31TyRcbyLcw+Pj46Me/JGkOVls43meJFF2JHjH8wb3TLz9cYOd/cglN9tNxAhAvAESYVM9OUe+fnLGpV5AF7jtHrz1tEk6fKMDbzfBcyiECSdudj7Vx8COVpwIs5WfBsgIPsAjIcAsfEBO8/Yh3vTjwEfSvRPUI3ohp0SK9fLkkcYZGw5aDGDIDul9x9d3y38qUFyigtT/EkIKsPkJTyduzz/5X9ipLfG+pvOVH3NS54JwKAXhsA3GKmh6k8S7JGIHVTIHfNEvm3P03cB0AroDYcUeMOooTjnQCNDfZgsSBcmu7AtndBEu2Sjwe5uy96wvEg9F9sYfZTQCQc06F2pFw86Qe0cVhfdkI4Sb4MWKVCvhETZwyaSBEFyk8BWcpAAhEMG+iJ+pve4rE7nVhPcdSKVP0DNEMMhb7g3qIN+Rwb2ysArxfNyuZzA8sFV8vI0AJztAd+ovUHNgIJwBotufRZSRhjo/7NMTYkfyc8bIO33JYj/RfJvCSOXOXJEciZNxUT9ExjrhvGJwgQQ1HJ64w0Fbirp2PouTdfUlzD76a9q/urn56L29urkC+QMcXhUIh/QDAedFsH7ep1PIPXNHCGhifgxsS9BHAtsJ2sWEiw9/lqEgBYYPzTagKgKC/EJF4GIGNEEC+h6KNORY7CGjFSdOCGMc2AlPakdsZRizXczg3Bcfu8RpXBTQR3Xy6mZW/sGMUfqx+RTqhAtcxoYGud+g0xZzU/bJ4Vs8GrrDE9jj8dAdX+7bY94752oPgFvS3EMO4AFONqCVp0mwnecOPnKJfeV/eHf1tgOQekw4e9aR0HCBfZuWEqQPgj0P/cjh/fDfU+pvYGP7C9C+AxBU8W4YIBi3bBJ7jz9P8Qd5/VrOWnu9g7c7+8vAe5zC/9qTHTzZKU8U5jlVPosWgkARn43rdB68dQharPfz4w6kgDfDH76Lp3HHLQO3kSiUfykwjBRUgVz0A0AMyhMNl6u8NRjRaxb6GYK4gnku8UOQ82dt4Ri2qEs0VtkajMov3IILtO5eORhgAnZLjCl8Q1Xx2XFb2JmLPIMTfQqgy2sLBJsGtCdgPY89TtBoujCIYFdWBcm9CS3kgl8BwFiP1LgLMScsttl2AXpgX4dirG+f9FIRg1Ki9pQ5+jAqCZtvZpUnvvGkltJNejYB1RO3SsLmG07P5lM7cZskrL9tomeNao1udhJW98V4ZedkRpMqN9MnJDma8VRwNe1pLWfTh9R1l6nxt9K6I2XHGskp8BYxEGOhhXvwVJFMjdzSFVbMiGkmF+fueVvFxNismo1q2qTK6XGP9NWJQ+PE9MlTiBodGp5By6DSgGAGNbmFCbqLovXr+TbLk7WXbDLdsIMnDeZoi67CNB0G5ycDet7vnyzGF/6MtjBN2wBvMFPbdMfNPmNaKPwcjuVeV61UD4YQm22cWdQj0IeCHRTXqtHqSNv0mj66BqoRVE0Pi3+vxtRsBmBXE0FA1fex+V9qpNcr8lm1hAp/XmlClP6nHWwdmSWgIQ977JyrYKTjCZ15qvmiu6GIU3iMcEIJqNOpdDtxOKaz6VY4el4zMXPX+TX5dH0FXIpuMrLNYMpqZ7tHjWv16OcS85beuUUKZo9cYF+VgBb8TQ9EuOQMjOQ9MMS2ESrOi3k/Tjy0k8OIevw83PI2MajKd7wTHhBPura4ts0eMZ+yJ0SzbOB0hM12yW22M3c0aHVc0JCYHFmEMRAsjFr+DWvpltRSWKYF0ezev//wq4x8QGQwenoKAyq8FwhFeJlnz4XrUwqMe3gfbdfAbKV/Mi6sYM3yBWpBIOjojZ5Lty6jSrSHSyc1NABxgd4IZpv0mdMC0JT6cixsts5otEN6TQt/7DZGVYcIv5aTJlE08+ePgtjYncEWPY6lOxu9Bcs02W6I6lIP4yznngcwJREKWpL8AAD5sisCnCd37yoylvnaxRQlLByBqXqrJAoIIzmr6wm9TgVnaKu6VmHw3QdGB3NxjKl1yo5oO6c5sTg4XGLoh3U3R2xDme6am2Sy8nG25FgHtDhW2YrcSaeY8fQvxce/utXOql7zF+WPv3aOYWHSBc2uFYT3SNJsoXJLvxDbjzBgFCc73v8Qg0T/LXOFWzi/N99s7122BwDwnnmsvJz5cW4nLhnc3UtAjjhLvwcSRJzwk9sBKn5KUE/MEnTgcE3ojJ338/PSTdp43KXWp6g6RFPxiObtm65L96hdCZpa1KJuex2xUBAl2RbE9515LBQuldI191R5cCI4ljVehfg0e7tkctekbeGlAk1fc67s84tYP/aXYP5YFKzG1kKnGvvj02B42e+fDc8uF8NGnaoZnk2Nau7BPOgnTE3GX8Oz8m70TdnjA++gYLblP+k+n0ohyEWifNwpARrypebWs2J8vwJBggrCvQeyfwMMGlrC4cruuXaiOjr5SeU3x8zmRpYC+6wCwz5MbjLGot7xFspKIcqQK9EYz2Wg6AN1LGyKDutFn+9Gs74mSLyBGda4Fho0uqlwxXvMH+lUvSLa7IzDLA/pps973wLrxv/ulNtNEuTPGzqFNuyWs2Ppv5cYujoxtF+bejcovWmzbQhnXuGezuNOxAUsw4Bdv7B7l+Kq0aA+paeXzB6MDWRU5vHrTKXlrTrGHT9dZ/x0nY/UuIxWh8swPqYVN4PFtVfFz9T20NLXctymlmeWja2h1GnN81oI1tnXv7KswQgHmcoPtqbzzRT+N94I9Xh0wu80RmcD97zlrjHyQDYkFfZNmgC+Mg84hwc6JYVfUQIiJXM6Db20lnC28NarsQO/CPNQw+Cnm0tOzvUc5VRVem43AW5szTapg6LEVahf0a48fmEHPCNfJQGZTskx9ZcRHR8bBF1dKGvmrWgECpM2Go0OGi9YwOasjoWmM+JhULB1F+542HLvDhxxnW/2Lg/aqGsr3fwH7AACZUzKYPbGZBskhUVKcPdT+Qwobck42iJxjjmWULrtmZu3iPvHCnFxk0rVPOTs8CH/KK/u/02f0P6RyvZMbMj4DV1+oBwAqziQN0qO6U41BBy5xtsipAc0Q5uu4NQzZGgjfNvnMF94HGzX62cPjm+WpMbojr5fXwoW5RJTVD5IGWr6g6X4bMBBp/bO4m93onxm6JdVdFTHtMmekhQ09b+os8Mrj7+2oJ5j91B6cxtIp5VMa5KYTd3qlbDC8DRZSDP7NzgI2XPoK8C4PwzNITzUoC+j3sOdJRme/wLgLpn7M3njq66jfMH588nlmPHn0/FQC8VspRGVpLVpaUQIKdPOlmAhK+hJoimY1VJNRT4a0BREQVB6jkAJW4L9ULTZhb7mXcSHbGGAJ7BMIn9O7wu/Va1Z4gi/kvR+lVYJizAto3rELDvckMkxug0vv9GPhM7LzXOeUngRLmMfuD1VDBQdIV/BTtkDsN5c0Tr+7VstGsk1mSpt1mVaLJLLesGmkVk7xcw77sHWuMqxuVdoOOBn8fTSvXyB5X+4gfJLjZQ6Q6X+mNdEANTZLM3U3AjMui6nPgBhL53UbfEBVk5HBP0Nz0bc43928QK229bL05ort2PHnymPOeWIIqXQI/4s2VHu4hGcUOHW8QqYdW4Evb2qZ7pNPB2viTjz/rvz8bTgje3+2TjoS2GVLhpr9MFLAO3l4L+EiL+aS+p/h8GfnZ5wv//w/GDv098Wg//lnP2rsfSDPFIv5dfnMnYFdm548s3VZO5DaaUm602/AhPcA7CeF2od/w8pkG3W9bfBX84vuJ/04mz4d65A1h+AA9lMMzUfzm3sHfaSyFdkRRc8xWZ4ORr//yZ/m01usBL27vRXshL2X3mjKxvGxeBEdvHV4t67oYu4/B6c0ovR+Wm/P7o8pfTytMXldxPQhhvwpm7soo7lRcHP4UjJny46sXiNKqGLLEqPZ28uWHzzfLWNH+G3CEcT71hQWoffI1MeyCK20x5lSvz5PGHSIsI84JJC/hldcpkluIrBZHFyPCqcuYB4xFqmiJhbPbQtu7tl99p69J0ZTNcnn/zlErOeAAFKFF7pkuKhT/tC51zytEILTssFLqNLfpXpCUkylLAMgsqfkmJRLBcpXG6TbRYpHjMwfSi5r+R2NwTg3WOGTrkN1eQn/HejRZxlTz4uN5abmClJbJjfyDLyz93xgHTPxu7J+QE0pcp0jkO38hlPst7j7ohUr3nYNL15nJf5KK/IT5iJ/pT0WGAarMbfbNLEn69IjmFDnICYac33Ty4QX7O97JfAQpc8sCzGYlIYRKPk37DNDslvyMOkfMp6YmsvDKS2ot6o8zfZbXin93lo0ecB+hgjwWbxjG6aejycus8CjMTDQEQZ3cop3eEl39CYr4jXY2ljBmbgbAOVInpYUCLGYWICd7V/SLpTFTvl1WOr+T0U8/uuZnoPfHoy4dk6P3i5C+HE1MzxgfQsc0Qd1TLeZ6AuWBXytAcLrMomMWpkgjdzkHTwVqOrZiUKGleiDE3MlFiptleh4T2J4BMOR9uEwIFnlyFA9pWsnMHE4Fsu4Lg+PNk1+48mROOYRopN603WSbBbVYvYsR5UoIuTAQypipd6OEPb3TAZmYjcyKtSh6ltPFea0DgQn2bJNlZialii6C17Az/ujOGBO94o3KeAjVz1lsFnoO8mJefBWyjzqoyHmv9mygcnM7rANHQuVrO8L5k4cbjc4OoGz4oyQYEYCdg8XM7WO33k4HoWCwhxnl0lGaq+AZwPlrPXX2tssLIXYUmpDaywU6x1YtcWBXepvBPBDXsGeIABvmuELziDBX7Fgj2EARyylKZpsPHE5li6Aw5CPFQPQgM7CN+kDSev4557JibOXCjTXXXJj6nCpIvHyGnHP/SxG9bHWUxHdZFfo3oyNBmhpgAwRYwrAV3rbEtWYSgFLhlUBxtNmjJAxJnW+Kqib2kTKNTGqTIfzc+kTPM3xcwm1WQ1BVK5GJv+xVc17Kh+GBvCbf4bY5/2qnrdptZyxjbV8E6dnakU7jf6yuidBjNPayQMu9mIXp4Nzvv9xelwdj4btDDsdDANppzekGnal8xdir/GIxnh7mXbWZ7681wYwVjZqTgo3gKEmcjUUgrOuGSTZGzLtada5l6R9Fe2RSQfdb/HDg9h7vD0E89Pl5iog4Wq0Ll4fAcHgE1sbwSK01RsqP0dR3PhqPZwfvlE7BVVzAjcrtAPPs/Rrkx5vSysHvWrrFqnA60vwZEQW71Q1nmSKc4c3L0FCffEsWVcsxQannEmd5YZ7AKQlodtqX1V7Q08j5UQAe2BlSsSWX2yBlOHOYdpkTXy9rqH08KQc6wxJlK4RUYow0Mf2Ca6dTiggvzI3E/TZ76GAU+cizH/iIUmc5+DyP8eyCCUMBWFtDgonAbMZR3RTHJWpgMpcYsZ3xRHermI9J2RnR8Bj9Mz5tlJEU37fn4rk8f78MARHQrwGCjGnBDT9lF5vKsYyMgSLuYhJ61Qqi1fRVbjmDrFVFxi+6g6+IuSGdOyJX/bqR5f13YWXe1gMXQccVxjinKULMEun2WwJFhB6Efhn6mjhXDx19yzD+fyvXhywx+4WluMJpYZRjzMK0mVwkXvQTyhxn5Xli1ixYJGLFrr4uRSc4vdXP/0AdMkPjEm7PzxT28S+IAglccfwi8hnID3yacr82mnEjcdxmEucjeq0c7spRqSKz1eZWStsEdVPmBhGLKcT8VzphsYr2RdPqd0b3U5vI6o1fdrJXcw1LJb73G2fLc8mQ5wD8b2nJ9zPqtNkvCySWiwP8bJU9zHcAcNTp0Dq1oTom9EU9bWqyvrVEmee1fc7x0pxuxbIBCs5MVrqzAOA5y2LLjGXVZEBrgZ+2XJNYJRqvcFPDqxGmUeZh5WD/SQF/KobrxfQjK8HLvD029Dhq8IBe3zuYdcludtVpcFuxFvqbStaZJ5rKHHDNkdK0bjGOGH8S7rv/nx480PH39657199+bHt++8dz9+9t784d2bP3o/fLx59+lPV+871dhunlrkGYMK4wXey7RfDE2vRqtX3Bs3PG+SwZJRMFGSbHieDPfbAhVWK2KagGQBzJKW1cAaGcfIJJF69HhRwQo0tf4lli2MAt6LpVRGVCRWKmm0PNinAkcpNNnX09RJTFlxQv+R18ESRk2P5dYugMxMUBs/X2FirwISs7+en7Acp9ZWBNE++SnmGDrHb1m5LRzTvnkTdpGdbTcYBQ9847hRfTp+qvPza4H3BXnWEMyU/M4HA92gZaGgYoC3oGa+zmmVuEDfCDnXkE+WIMF1Mj82AR275O2731399P7G+3D1L5LwP9+8u/5scnc8Rkwl545drtd7YCdFGY7kqe+5vc4z2/lkZMLOJWMQw/F45I5Ovg2HsJp3TDRxfos8PXOO7E6QIjFM3jqBWbIFlJpVWg2B9jsQWHqN2IIqlJJ0ePWCbl9Z60eU3eOKsAouWdikYk0VV4pXAsJ8Bf070s6KduQ6PHFeLRcK8NETXXYXUlGcaJYPrQo7S7FXdlXEiz+skici7Tc917/IyRdZ/kkUoQmAufjO7FlWCRR3YwofEbo1q1Irl67FXCN4VzAPADRPNs8sQp6VZelr/LrMLBQHZOVnYmN58mj1nBdsvMq61YCXjjXkrKQWeV7bUlfFJVYBaM8Zqhm76j6Ty0a7im1n1jZWkIESwSkGe5NmcctMn1INenGazz5NStQlQlO/OnYEfQJ/woJl/kzTJHOqtZhJR4YOPTTFDjETQ8w4m5p46DgdXW27QXOZn3qsNfGUkeA59te8EAZqcHilvAZGgkewCHNFWmTTIrzKQF8F+c/o/uuVKq0se4jBuFipeu5v/HmYPwPJPsGJ8nNiHIZSwdWDlaAZK4eBuIxoXDlDZSYta8X5/Gg4OsW6pd3R8OLEHY2/kUWiXKBYPEdF9JWpstnUMG07Oe/mdPU8ox6YI7MwkBVeXG7FimqJTkHCHcvVG9OPMx+TQ4NCOkr3nzlbVtBknnvVTplTBW1iQNKJy242i4oMvdpDbMmdlPUc5OP9QFSmUqYQNfVQk9WwvpnKUNySIVhuHJz22LR5vGRYgWLJ70O6nSO2wHqXfA20N0Cpw3tDl5cjXh6cw0M/1VudDIvo+GBfO1q9JWCwIavkXyt+asJdEeFOs9AK0Q+mV3fiLi2n0+nUl/M64OasYX4ta4dVVA5eiMcKVEF0w0VevTBsQFcLX3czrpuGdskBwXz151Wl0s7R/uMlNe8yhCxNNgkosEw1LTiIzRhg4k890dYWNf467pMbXYxP3OEFCMHLwak7/kZCcA1Lol7pYRaapvEYixVhFZ3Dk4aUEMU8AVvfs91mKjFNdYHQt0o50uaQ6VsFGitGvmEeJhbD5cdLKkOjTZuvZG5Teb/mYUk10f9W/DZilbjLhV8HYDGZGaWxjK7iX0bhib84t0X3iHnfCoqBCHszYb/Bb3iotHfyFVAhVzNs6OQ3vrhgdrHy9pqVjbW5ig6YN17CmCCKynxKZKEZUVhE8fXFaiqVAtVqgdoiKSvsJ14IMPyWo7i7lc1NaMq3XtRfPLuV0oYmmGJQA5f9o5oIPjjLkmDK+mu2W2ezoQYPybYM9BF3+dC8oEFbqMLLIo/snMu6nvpYjbqFNQSqKHFMh4R+TV4wAf2U27hGH4snxoGFj99aUeGSmscFD7kz1KAGllUM3rUOblmWS2pfqBNojLFHJiczXg7JLGiRm9dP/R2N1JI026xQDModLRwLh7tVyEv9LGZyp23hKOtU7LTeSLOYsh0reFDt+CiK+KuhHK/Idf3X2gACMW5ciVCVIaHo9u5MyAq0n17uh5EKEJcVL8W3VJWeO3n/LKtr4qU7O0RMCjCVZDxAlWRMuuPhaPBNHbA1UQ1MD0HWwTW6SfmRdNvTa3lIkAaXUTIrvFj/8CIFh6lyhSuso0txx0or1ntiw+wN6I5ntGXexn+OEj/oVFLdeqTBJ9Z+aERsjdWtBHeIem4zwFjH1rp96TfNL1l7CpsNw0MQ2z0gTcfWus1WNNnODYnZh25St9ncsaPSrexetzk9utUO1hi2+zMav/Jefo3d+Vr70HBgXojyX1phoWUSe4vtaIPnl6DYwNPhKOLC6YwlMI6HZydqFvy3EE1/7yUbG50t/3N1G9kuGg/raziWLj+zmGNDfKuiYL1+8L+85hfOlhjXuobyGxEWdDw/HfT7F/MhnS1mTXGutaBssa61jRlNYzXR7ljUFL1+f/Xm3R9+fP/23Sfv5sc/vvvo/fAWzl9vyKPJGl3Rjd7nCYuleMsefqgi2XSZZxOifXWd3f8svmmv0BJKN7PIJlEdyqqnxezYFFSaN3+RniKcDhy+btSjbotxK9/fp9R0myXBs8MdiBGNl/kK3ZImgjuTSjC317S9++Bxj+IJ+7pH9lOETrcjGTZnV//iPxFaSdqFVqraYhlP6bQPvWwRsElESKaFvvt78c0Cu1mEOcqOLvy8bIcjtqft0KhNwjhCe46Yfqdad9YqNK6cIUHsGf3ZzN9qoNueiA9+x9dXhlKI6XJbkVnfImO2iN4QViT7sgL6c59omVYvmI31XO+dmltOiH1ZLptqzTdByqniTGW4SmVb7uUXcXFflYwv4aOw76tkPVLhIGRuovKyAHUzjQp4K56nLooqml36rArj7YCXmb4c4xG+PBHFYlpSn/2+GBEKwtfR3iIjBCy08v5g6o+ylaSnrKRd2ZF5wu70co9fq00HwnYWPM+yIut1rfxqjA/KtzQoQUBKDj5Zb7NcfBc1D39iebwY3TTRvomERx/1tNHu0WdapoYXoVSCFgRie2TYl6D+laYJj0Yuvo+ZBU8EYSC/utioNl+kQhBH+HJcCaz4Jk8emclmiAGXPHwKv4McF0ZzQaDcLcIDKQUo4yuZ9UAXhRo0ktVxUBJI95fSR/cl5NF9AXm4lhUzhk9kJs/RfwNWG3ONWoAAAA=="  # __EMBEDDED_PATCH__

# ---------------- 1b. GDN state-rollback patch (needed for lossless MTP) ----------------
# Stock vllm-tpu 0.28.0 never rolls back the gated-DeltaNet recurrent state of
# rejected draft tokens, so ANY speculative decoding corrupts outputs on this
# model. We apply a port of upstream PR #3178 (see patches/ in the repo;
# verified 12/12 greedy exact-match vs non-speculative). Safe with MTP off too.
if MTP_PATCH_B64:
    import base64
    import gzip
    import importlib.util
    diff = gzip.decompress(base64.b64decode(MTP_PATCH_B64)).decode()
    Path("/tmp/mtpfix.diff").write_text(diff)
    pkg_root = os.path.dirname(os.path.dirname(
        importlib.util.find_spec("tpu_inference").origin))
    p = subprocess.run(["patch", "-p1", "-d", pkg_root, "-i", "/tmp/mtpfix.diff",
                        "--no-backup-if-mismatch", "-N"],
                       capture_output=True, text=True)
    if p.returncode == 0:
        publish("mtp-patch-applied")
    else:
        log(p.stdout[-1500:], p.stderr[-500:])
        if CFG["mtp_tokens"] > 0:
            publish("mtp-patch-failed",
                    note="disabling MTP: unsafe without the rollback patch")
            CFG["mtp_tokens"] = 0

# ---------------- 2. XLA compile cache ----------------
cache_tar = find_input("*/xla_cache*.tar.gz", "xla_cache*.tar.gz")
cache_dir = find_input("*/xla_cache", "xla_cache")
t = time.time()
if cache_tar:
    subprocess.run(["tar", "-xzf", cache_tar, "-C", "/tmp"], check=False)
elif cache_dir:
    subprocess.run(["cp", "-r", cache_dir, "/tmp/"], check=False)
    subprocess.run(["chmod", "-R", "u+w", XLA_CACHE], check=False)
n_entries = len(glob.glob(XLA_CACHE + "/*"))
if n_entries:
    publish("cache-restored", entries=n_entries, secs=int(time.time() - t))
else:
    publish("cache-missing", note="cold compile will take ~35 min instead of ~18")

# ---------------- 3. weights ----------------
weights_slug = CFG["weights_dataset"].split("/")[-1]
model_path = find_input(weights_slug)
if model_path and os.path.exists(os.path.join(model_path, "config.json")):
    publish("weights-mounted", path=model_path)
else:
    publish("weights-download", model=CFG["hf_model_id"],
            note="attach the weights dataset to skip this (~5 min parallel download)")
    t = time.time()
    from huggingface_hub import snapshot_download
    model_path = snapshot_download(CFG["hf_model_id"], allow_patterns=[
        "*.safetensors", "*.json", "*.txt", "tokenizer*", "vocab*", "merges*"])
    publish("weights-downloaded", secs=int(time.time() - t))

# ---------------- 4. launch vLLM ----------------
server_args = [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
               "--model", model_path,
               "--tensor-parallel-size", "8",
               "--max-model-len", str(CFG["max_model_len"]),
               "--max-num-seqs", str(CFG["max_num_seqs"]),
               "--port", str(PORT),
               "--api-key", CFG["api_key"],
               "--served-model-name", CFG["served_model_name"],
               "--reasoning-parser", "qwen3"]
if CFG["mtp_tokens"] > 0:
    server_args += ["--speculative-config",
                    json.dumps({"method": "mtp",
                                "num_speculative_tokens": CFG["mtp_tokens"]})]
if CFG["tool_call_parser"]:
    server_args += ["--enable-auto-tool-choice",
                    "--tool-call-parser", CFG["tool_call_parser"]]
if CFG["reasoning_effort_default"] != "xhigh":
    # The chat template defaults reasoning_effort to 'xhigh'; ship a copy with a
    # different default so the server-side default changes without client changes.
    tc = json.loads(Path(model_path, "tokenizer_config.json").read_text())
    template = tc["chat_template"].replace(
        "reasoning_effort|default('xhigh')",
        f"reasoning_effort|default('{CFG['reasoning_effort_default']}')")
    Path("/tmp/chat_template.jinja").write_text(template)
    server_args += ["--chat-template", "/tmp/chat_template.jinja"]

publish("server-launch", max_model_len=CFG["max_model_len"],
        max_num_seqs=CFG["max_num_seqs"], mtp=CFG["mtp_tokens"],
        reasoning_default=CFG["reasoning_effort_default"])
vllm_tail = collections.deque(maxlen=200)
server = subprocess.Popen(server_args, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True)
def pump():
    for line in server.stdout:
        line = line.rstrip()
        if line:
            vllm_tail.append(line)
            print(f"[vllm] {line[:500]}", flush=True)
threading.Thread(target=pump, daemon=True).start()


def healthy():
    try:
        req = urllib.request.Request(f"http://127.0.0.1:{PORT}/v1/models",
                                     headers={"Authorization": f"Bearer {CFG['api_key']}"})
        with urllib.request.urlopen(req, timeout=5) as r:
            return r.status == 200
    except Exception:
        return False


t = time.time()
while time.time() - t < 5400:
    if server.poll() is not None:
        tail = "\n".join(list(vllm_tail)[-100:])
        log(f"server exited rc={server.returncode}\n{tail}")
        publish("failed", step="server", rc=server.returncode, tail=tail[-2500:])
        sys.exit(1)
    if healthy():
        break
    el = int(time.time() - t)
    if el % 120 < 6:
        publish("compiling", elapsed_s=el)
    time.sleep(5)
else:
    publish("failed", step="health-timeout",
            tail="\n".join(list(vllm_tail)[-60:])[-2500:])
    sys.exit(1)
publish("serving", startup_secs=int(time.time() - t))

# ---------------- 5. sanity + mini benchmark ----------------
def completion(prompt, max_tokens, stream=False):
    body = {"model": CFG["served_model_name"], "prompt": prompt,
            "max_tokens": max_tokens, "temperature": 0.0}
    if stream:
        body.update(stream=True, ignore_eos=True,
                    stream_options={"include_usage": True})
    req = urllib.request.Request(
        f"http://127.0.0.1:{PORT}/v1/completions",
        data=json.dumps(body).encode(),
        headers={"Content-Type": "application/json",
                 "Authorization": f"Bearer {CFG['api_key']}"})
    if not stream:
        with urllib.request.urlopen(req, timeout=600) as r:
            return json.load(r)
    t0 = time.time(); ttft = None; gen = 0
    with urllib.request.urlopen(req, timeout=900) as r:
        for raw in r:
            line = raw.decode("utf-8", "ignore").strip()
            if not line.startswith("data:"):
                continue
            data = line[5:].strip()
            if data == "[DONE]":
                break
            try:
                obj = json.loads(data)
            except Exception:
                continue
            ch = obj.get("choices") or []
            if ch and ch[0].get("text"):
                if ttft is None:
                    ttft = time.time() - t0
                gen += 1
            if obj.get("usage"):
                gen = obj["usage"].get("completion_tokens", gen)
    return ttft, time.time() - t0, gen

try:
    completion("Hello", 8)  # warm up remaining lazy compiles
    txt = completion("The capital of France is", 8)["choices"][0]["text"]
    ttft, total, gen = completion("Write a short story about a lighthouse.", 192, stream=True)
    tps = (gen - 1) / (total - ttft) if gen > 1 else 0.0
    publish("benchmark", decode_tok_s=round(tps, 1), sanity=txt.strip()[:60])
except Exception as e:
    publish("benchmark-error", err=str(e)[:200])

# ---------------- 6. tunnel ----------------
cf = Path("/tmp/cloudflared")
if not cf.exists():
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        cf)
    cf.chmod(0o755)
tunnel = subprocess.Popen([str(cf), "tunnel", "--url", f"http://127.0.0.1:{PORT}",
                           "--no-autoupdate", "--protocol", "quic"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
pat = re.compile(r"https://[a-z0-9-]+\.trycloudflare\.com")
deadline = time.time() + 180
lines = []
def pump_cf():
    for line in tunnel.stdout:
        lines.append(line.rstrip())
threading.Thread(target=pump_cf, daemon=True).start()
while time.time() < deadline and url is None:
    for ln in lines:
        m = pat.search(ln)
        if m:
            url = m.group(0).rstrip("/")
            break
    time.sleep(1)

if url:
    log("=" * 70)
    log(f"ENDPOINT : {url}/v1")
    log(f"API KEY  : {CFG['api_key']}")
    log(f"MODEL    : {CFG['served_model_name']}")
    log("=" * 70)
    publish("ready", endpoint=f"{url}/v1", api_key=CFG["api_key"],
            model=CFG["served_model_name"], max_model_len=CFG["max_model_len"],
            keepalive_min=CFG["keepalive_min"])
else:
    publish("tunnel-failed", note="server still reachable inside the kernel on :8000")

# ---------------- 7. keepalive ----------------
t_serve = time.time()
while time.time() - t_serve < CFG["keepalive_min"] * 60:
    time.sleep(120)
    if server.poll() is not None:
        publish("stopped", reason="server-exit", rc=server.returncode)
        sys.exit(1)
    up = int((time.time() - t_serve) / 60)
    if up % 10 < 2:
        publish("heartbeat", up_min=up, endpoint=(f"{url}/v1" if url else None))
publish("auto-shutdown", served_min=CFG["keepalive_min"])
server.terminate()
sys.exit(0)


### Launch 🎬 (leave this cell running — it IS the server)
Timeline: pip install ~10 min → weight load + XLA compile (~18 min with the cache dataset
attached, ~35 min without) → **look for the `ENDPOINT` / `API KEY` lines below** → server
keeps running until `keepalive_min` elapses or you stop the session.

Use the endpoint from anywhere:
```bash
curl <ENDPOINT>/chat/completions -H "Authorization: Bearer <API_KEY>" \
  -H "Content-Type: application/json" -d '{
    "model": "qwen3.8-27b",
    "messages": [{"role": "user", "content": "Hello!"}],
    "chat_template_kwargs": {"reasoning_effort": "low"}
  }'
```


In [ ]:
!python serve_qwen38.py